# 15 · Streaming delivery

**Structural demo, CPU, under 10 s.**

The specification is streaming-shaped: `Session.next_input()` returns the next event
*or `None` when no input is ready*, so inputs need not exist before scoring starts.

That is separate from whether a given driver *delivers* one event at a time. This
notebook walks the three things that get conflated under the word "streaming":

1. **shape** — the session contract (already true everywhere)
2. **delivery** — batched vs one-at-a-time (per-path default, opt-in either way)
3. **real time** — pacing against a wall clock (layered on top of delivery)

The claim to hold onto: **streaming changes *when* inputs must exist, not *what* comes
out.** Every comparison below checks that.

In [1]:
import warnings
# Silence only the known-benign xarray subclass notice, NOT everything: a blanket
# filter would also hide BrainScore's own warning that installed dependencies are
# outside the supported bounds, which is exactly what you want to see.
warnings.filterwarnings('ignore', message='.*__slots__.*')
import numpy as np
import pandas as pd
import xarray as xr

from brainscore_core.streaming_helpers import (
    StimulusSetSession, StreamingStimulusSetSession,
    WindowedStreamSession, RealTimeStreamSession,
    _drive_neural_session_via_process, _drive_neural_session_streaming,
)
from brainscore_core.supported_data_standards.brainio.assemblies import NeuroidAssembly
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

def make_stimuli(n):
    ss = StimulusSet(pd.DataFrame({
        'stimulus_id': [f'stim_{i}' for i in range(n)],
        'sentence': [f'sentence number {i}' for i in range(n)],
    }))
    ss.identifier = 'streaming-demo'
    return ss

class DemoSubject:
    """Stand-in subject: deterministic, and records how it was called."""
    def __init__(self):
        self.call_sizes = []
    def start_recording(self, region, time_bins=None, recording_type=None):
        pass
    def process(self, stimuli, multi_modality=False):
        self.call_sizes.append(len(stimuli))
        ids = list(stimuli['stimulus_id'])
        data = np.array([[float(len(i)), 1.0] for i in ids])
        return NeuroidAssembly(data,
            coords={'stimulus_id': ('presentation', ids),
                    'neuroid_id': ('neuroid', ['n0', 'n1'])},
            dims=['presentation', 'neuroid'])

print('ready')

ready


## 1 · Batched delivery — the default

The perception driver drains the session and makes **one** `process()` call. This is
the default because it is what keeps bulk scoring tractable and reproduces legacy
scores exactly.

In [2]:
stimuli = make_stimuli(6)

batch_subject = DemoSubject()
batch_session = StimulusSetSession(stimuli, record='IT')
_drive_neural_session_via_process(batch_subject, batch_session)
batch_out = [e.payload for e in batch_session.emitted if e.channel == 'neural:IT'][0]

print('process() calls :', batch_subject.call_sizes)      # one call, all six rows
print('output shape    :', batch_out.shape)

process() calls : [6]
output shape    : (6, 2)


## 2 · Incremental delivery — opt in with `streaming = True`

`StreamingStimulusSetSession` holds a row *iterator*, not a materialized list, so
events are built only when asked for. The driver processes one, emits, then asks for
the next.

In [3]:
stream_subject = DemoSubject()
stream_session = StreamingStimulusSetSession(stimuli, record='IT')
_drive_neural_session_streaming(stream_subject, stream_session)
stream_out = stream_session.collect('neural:IT')

print('process() calls :', stream_subject.call_sizes)     # six calls of one row each
print('events emitted  :', len(stream_session.emitted))
print('max held at once:', stream_session.max_events_held)

process() calls : [1, 1, 1, 1, 1, 1]
events emitted  : 6
max held at once: 1


/opt/anaconda3/envs/brainscore-unified-fresh/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
/opt/anaconda3/envs/brainscore-unified-fresh/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


### The property that makes it worth having

Same subject, same stimuli, different arrival. If these ever diverge, streaming has
stopped being an interface change and become a scientific one.

In [4]:
np.testing.assert_allclose(np.asarray(batch_out), np.asarray(stream_out))
print('batched and streamed outputs are identical ✓')

batched and streamed outputs are identical ✓


## 3 · Windowed delivery — batching without buffering the feed

One-at-a-time is correct but collapses the batch to a single stimulus. A window
restores batching while the memory held stays **constant** no matter how long the feed
runs. `frames` is any iterable, so a lazy decoder stays lazy.

In [5]:
def frame_feed(n):
    for i in range(n):
        yield f'frame_{i:04d}.png'

for n_frames in (60, 600):
    session = WindowedStreamSession(frame_feed(n_frames), fps=30, window_ms=1000, record='IT')
    subject = DemoSubject()
    _drive_neural_session_streaming(subject, session)
    print(f'{n_frames:>4} frames -> {session.windows_emitted:>2} windows, '
          f'{session.max_frames_held} frames held at once')

  60 frames ->  2 windows, 30 frames held at once
 600 frames -> 20 windows, 30 frames held at once


A 10× longer feed holds the same number of frames. That is the whole point — and the
window carries the timing a benchmark needs to align it against brain time.

In [6]:
session = WindowedStreamSession(frame_feed(90), fps=30, window_ms=1000, record='IT')
event = session.next_input()
while event is not None:
    m = event.meta
    print(f"window {m['window_index']}: {m['window_start_ms']:>6.0f}–{m['window_end_ms']:>6.0f} ms, "
          f"{m['n_frames']} frames, partial={m['partial']}")
    event = session.next_input()

window 0:      0–  1000 ms, 30 frames, partial=False
window 1:   1000–  2000 ms, 30 frames, partial=False
window 2:   2000–  3000 ms, 30 frames, partial=False


## 4 · Real-time delivery — what happens when the subject can't keep up

The feed advances whether or not the model is finished. All three answers are
legitimate and they measure different things, so the policy is explicit:

| policy | behaviour | measures |
| --- | --- | --- |
| `drop` | skip what arrived while busy | live operation; bounded latency, lost coverage |
| `lag` | process everything, fall behind | offline operation; complete coverage, unbounded delay |
| `error` | refuse to continue | a benchmark only meaningful if real time was achieved |

The clock is injectable, so this runs instantly instead of sleeping.

In [7]:
class FakeClock:
    def __init__(self): self.t = 0.0
    def __call__(self): return self.t
    def sleep(self, s):
        if s > 0: self.t += s
    def work(self, ms): self.t += ms / 1000.0

def run_realtime(policy, ms_per_window, n_frames=300):
    clock = FakeClock()
    inner = WindowedStreamSession(frame_feed(n_frames), fps=30, window_ms=1000, record='IT')
    rt = RealTimeStreamSession(inner, policy=policy, clock=clock, sleep=clock.sleep)
    try:
        while True:
            event = rt.next_input()
            if event is None: break
            clock.work(ms_per_window)
    except RuntimeError as err:
        return {'policy': policy, 'raised': str(err)[:70] + '…'}
    return rt.report()

print('fast model, drop  :', run_realtime('drop', 100))
print('slow model, drop  :', run_realtime('drop', 3000))
print('slow model, lag   :', run_realtime('lag', 3000))
print('slow model, error :', run_realtime('error', 4000))

fast model, drop  : {'policy': 'drop', 'windows_delivered': 10, 'windows_dropped': 0, 'max_lag_ms': 0.0, 'realtime_factor': 0.1, 'kept_up': True}
slow model, drop  : {'policy': 'drop', 'windows_delivered': 4, 'windows_dropped': 6, 'max_lag_ms': 1000.0, 'realtime_factor': 1.2, 'kept_up': False}
slow model, lag   : {'policy': 'lag', 'windows_delivered': 10, 'windows_dropped': 0, 'max_lag_ms': 18000.0, 'realtime_factor': 3.0, 'kept_up': False}
slow model, error : {'policy': 'error', 'raised': 'real-time budget missed: 3000 ms behind at window 1 (window is 1000 ms…'}


`realtime_factor` is processing seconds per second of material — below 1.0 keeps up.
Note `drop` and `lag` see the *same* too-slow model and disagree about what to do:
`drop` stays current and loses windows, `lag` keeps every window and falls behind.

## 5 · Behavioral trials, one at a time

Behavior streams too, and here the important property is not speed. A behavioral
readout is fit inside `start_task`; putting that in the trial loop would refit on every
trial and silently change what is being measured. It runs **once**.

In [8]:
from brainscore_core.contract import TaskContext
from brainscore_core.streaming_helpers import _drive_behavior_session_streaming
from brainscore_core.supported_data_standards.brainio.assemblies import BehavioralAssembly

class BehaviorSession:
    def __init__(self, ctx):
        self.task_context = ctx; self.streaming = True
        self.requested_output_channels = ('behavior',); self.emitted = []
    def next_input(self): return None
    def emit(self, e): self.emitted.append(e)

class BehaviorSubject:
    def __init__(self): self.start_task_calls = 0; self.trials = 0
    def start_task(self, ctx): self.start_task_calls += 1
    def process(self, stimuli, multi_modality=False):
        self.trials += 1
        ids = list(stimuli['stimulus_id'])
        return BehavioralAssembly(np.zeros((len(ids), 1)),
            coords={'stimulus_id': ('presentation', ids), 'choice': ('choice', ['c0'])},
            dims=['presentation', 'choice'])

subject = BehaviorSubject()
ctx = TaskContext(task_type='passive', metadata={'stimulus_set': make_stimuli(5)})
_drive_behavior_session_streaming(subject, BehaviorSession(ctx))
print('trials delivered :', subject.trials)
print('start_task calls :', subject.start_task_calls, '(must be 1 — the readout is fit there)')

trials delivered : 5
start_task calls : 1 (must be 1 — the readout is fit there)


## What this notebook did and did not show

**Did:** that all four channel families support incremental delivery, that it produces
identical values to batching, that windowed delivery bounds memory independently of
feed length, and that real-time delivery forces an explicit policy.

**Did not:** score a real model or a real benchmark. The subjects here are
deterministic stand-ins and the "frames" are filename strings. Pointing this at a
decoded video with real weights needs a GPU host and real model weights; it is not
part of the laptop path.

Nothing here mutates global state, so there is nothing to reset.